# Size (SIZE) — Risk-Factor Construction Spec (USE4-faithful)

**Purpose.** This notebook is the **source-of-truth spec** for building the
**SIZE** style factor exactly as MSCI describes it in *Barra USE4 Empirical Notes*
(Appendix A, p. 52). It is **not** a research notebook. The deliverable is a
clean, USE4-faithful SIZE factor written to parquet, suitable for input to a
multi-factor risk model.

**Audience.** You — plus whatever AI assistant you are driving. Treat its output as deeply untrustworthy until audited. Follow the stages linearly. Each stage has:
- ✅ **PDF SPEC** — exact verbatim quote or paraphrase from the USE4 documentation
- ❓ **NOT IN PDF** — something the PDF does not specify; a judgment call you must make
- 💡 **DEFAULT** — a reasonable default for any ❓ item, chosen to match standard Barra practice
- 🧪 **VALIDATE** — sanity checks to run after each stage

**Inputs:**
- `SHARADAR_SEP.csv` — daily SEP prices (`marketcap`, `closeunadj`)
- `data/out/estu_monthly.parquet` — monthly ESTU membership and market cap

**Deliverable:** A parquet file `size_use4.parquet` keyed by `(permaticker, signal_date)`
containing the standardized SIZE exposure for every stock in the coverage universe on every
monthly signal date.

**Companion specs:**
- SIZE is consumed as a regression weight (`√mcap` ∝ `exp(lncap/2)`) in the WLS factor
  return estimation step of the risk model. The raw `lncap` column must be preserved
  without transformation for this purpose.
- NLB (`nlb_spec.ipynb`) orthogonalizes to Beta; Size plays the same role for
  NLS (`nls_spec.ipynb`), which reads `lncap` and `size_score` from this deliverable.

## §1. The USE4 SIZE spec — verbatim quotes

### 1a. Barra USE4 Empirical Notes, Appendix A (p. 52) — formal descriptor definition

> **SIZE**
>
> *Definition:* `1.0 · LNCAP`
>
> *LNCAP:* *"Log of market capitalization of the firm."*

### 1b. Barra USE4 Empirical Notes, Section 3.4 (p. 16) — economic context

> *"Size is one of the primary sources of equity return covariance, capturing the
> large-cap vs. small-cap return spread."*

### 1c. Barra USE4 Methodology Notes, Section 2.3 (p. 9) — standardization rule

> *"Descriptors are standardized to have a mean of 0 and a standard deviation of 1.
> ... μ_l is the cap-weighted mean of the descriptor (within the estimation universe),
> and σ_l is the equal-weighted standard deviation."*

### 1d. WLS usage note (downstream, not relevant to this build)

LNCAP also serves as the regression weight in the WLS factor return estimation step:
the weight for stock `i` is `w_i = exp(lncap_i / 2) = √(mcap_i)`. This is why the
raw `lncap` descriptor must be saved without modification to the output parquet — the
downstream consumer needs its absolute level, not just its cross-sectional rank.

---

**That is all the PDFs say about SIZE / LNCAP construction.**
The PDFs do not specify: the price column to use, the shares column source, how to
handle non-positive market cap, the calendar start date, or the outlier treatment.

## §2. End-to-end pipeline

```
┌─────────────────────────────────────────────────────────────────────┐
│  STAGE 0:  Setup, imports, paths                                    │
├─────────────────────────────────────────────────────────────────────┤
│  STAGE 1:  Load SEP prices — marketcap snapshots                    │
├─────────────────────────────────────────────────────────────────────┤
│  STAGE 2:  Build ESTU per signal_date                               │
├─────────────────────────────────────────────────────────────────────┤
│  STAGE 4:  SIZE estimator — lncap = ln(marketcap × 1e6)             │
├─────────────────────────────────────────────────────────────────────┤
│  STAGE 5:  Outlier treatment  (4σ trim, ESTU equal-weighted mean)   │
├─────────────────────────────────────────────────────────────────────┤
│  STAGE 6:  Standardize  (cap-weighted mean=0, EW std=1)             │
├─────────────────────────────────────────────────────────────────────┤
│  STAGE 7:  Save deliverable                                         │
├─────────────────────────────────────────────────────────────────────┤
│  STAGE 8:  Validate                                                 │
└─────────────────────────────────────────────────────────────────────┘
```

**No Stage 3.** SIZE does not require daily log returns, a risk-free rate, or a
cap-weighted market benchmark. The descriptor is a spot computation from a single
date's market capitalization.

**PIT discipline:** For any `signal_date t`, the only data permitted is data dated ≤ `t`.
The backward asof join of SEP prices to the signal calendar enforces this.

## §3. Output schema — what the worker delivers

**Raw descriptor column:** `lncap`
**Standardized score column:** `size_score`

**File:** `data/out/size_use4.parquet`

**Compression:** zstd, statistics=True

**Schema:**

| Column | Type | Description |
|---|---|---|
| `permaticker` | Int64 | Sharadar permanent ticker ID |
| `signal_date` | Date | End-of-month rebalance date (last trading day of month) |
| `in_estu` | Bool | Whether this stock was in the estimation universe on this date |
| `mcap` | Float64 | Market cap on `signal_date` (from ESTU join; used for cap-weighting) |
| `lncap` | Float64 | **Raw descriptor** — `ln(marketcap × 1,000,000)`, natural log of total market cap in dollars |
| `size_score` | Float64 | **Final SIZE exposure** — cross-sectionally standardized (the deliverable) |
| `n_obs` | UInt32 | Always 1 for valid rows (spot measure); kept for schema consistency |

**Coverage rules:**
- Compute `lncap` and `size_score` for **every stock with a valid SEP price on or before
  `signal_date`** (i.e., `n_obs ≥ MIN_OBS = 1`), not just ESTU members.
- Standardization moments (mean, std) are computed using **ESTU members only**.
- Non-ESTU stocks get the same standardization parameters applied so the factor is
  comparable across the coverage universe.
- `mcap` in the output comes from the ESTU monthly join and is in millions of dollars
  (consistent with all other USE4 factor outputs). `lncap` is in units of ln(dollars).

## §4. STAGE 0 — Setup, imports, paths

Standard imports. Use polars, not pandas, throughout (project convention).

✅ **PDF SPEC — SIZE parameters:**

> `LNCAP_WEIGHT = 1.0` — *"SIZE = 1.0 · LNCAP"* (Appendix A, p. 52)

❓ **NOT IN PDF — minimum observations threshold:** The PDF does not specify a minimum
data requirement. For a spot measure with no rolling window, this is a trivial decision.

💡 **DEFAULT:** `MIN_OBS = 1` — any single valid (marketcap, price) observation on or
before the signal date is sufficient. Missing data is handled by the asof join itself.

❓ **NOT IN PDF — calendar start.**

💡 **DEFAULT:** `CALENDAR_START = date(1999, 1, 1)`. Matches all other USE4 factor builds.

```python
# ───── Parameters ─────────────────────────────────────────────────────────────
FACTOR_TYPE = "fundamental"   # read by /build-factor — no daily prices needed
LNCAP_WEIGHT   = 1.0            # ✅ PDF SPEC — SIZE = 1.0 × LNCAP (App. A, p. 52)

# 💡 DEFAULT (NOT IN PDF) — spot measure; any single valid obs suffices
MIN_OBS        = 1

# 💡 DEFAULT (NOT IN PDF) — calendar start
CALENDAR_START = date(1999, 1, 1)
```

## §5. STAGE 1 — Load SEP prices (marketcap snapshots)

**Factor-specific data stage.** SIZE does not need daily log returns, a risk-free rate,
or a market benchmark. It needs only a monthly-frequency snapshot of each stock's
market capitalization.

✅ **PDF SPEC:** LNCAP = ln(total market capitalization). No further data specification.

❓ **NOT IN PDF — market cap source.** The PDF says "market capitalization" but does not
specify the data column or how to derive it.

💡 **DEFAULT:** Use the `marketcap` column from `SHARADAR_SEP.csv`.

**Why `marketcap` directly rather than `closeunadj × sharesbas`:**
`SHARADAR_SEP.csv` does not include `sharesbas` (shares basic). The `marketcap`
column is computed by Sharadar as `closeunadj × sharesbas` and is algebraically equivalent
to `close × shares_outstanding` as required by the spec. Since it is pre-computed and
available daily, it is the correct and only practical source.

**Unit note:** Sharadar `marketcap` is in **millions of dollars**. Therefore:
```
lncap = ln(marketcap × 1,000,000)  =  ln(marketcap) + ln(1,000,000)
```
The additive constant `ln(1e6) ≈ 13.816` shifts the entire cross-section by a fixed
amount and has no effect on standardized scores, but it is included so that `lncap`
is in correct units of ln(dollars).

❓ **NOT IN PDF — non-positive market cap.** Should not occur for ESTU members, but
Sharadar floors `marketcap` at $0.1M for some OTC names; and deleted/suspended tickers
may have `marketcap = 0` or null.

💡 **DEFAULT:** Filter to `marketcap > 0` and non-null at load time. Mark `lncap` null
for any remaining non-positive values after join.

**Load instructions:**
- Scan `SHARADAR_SEP.csv`
- Select: `permaticker`, `date`, `marketcap`, `closeunadj` (for sanity check)
- Filter: `marketcap > 0`, `marketcap` not null, `date >= CALENDAR_START`
- Deduplicate to one row per `(permaticker, date)` (keep max `marketcap` on ties)
- Sort by `[permaticker, date]` for fast asof joins in Stage 4

🧪 **VALIDATE:**
- Total SEP rows loaded ≈ 30–40M
- `marketcap` range: 0.1 to ~4,000,000 ($M), i.e., $100K to ~$4T
- No null or non-positive `marketcap` after filter
- `ln(marketcap × 1e6)` spot-check: `marketcap = 1000` (=$1B) → `lncap ≈ 20.7`

## §6. STAGE 2 — ESTU monthly snapshots

**Identical to the shared USE4 build infrastructure** — no factor-specific decisions here.

✅ **PDF SPEC:** Standardization moments (cap-weighted mean, equal-weighted std) are
computed within the estimation universe on each signal date and applied to all stocks.

**Load instructions:**
- Read `data/out/estu_monthly.parquet`
- Columns: `permaticker`, `signal_date`, `in_estu`, `mcap`
- Filter: `signal_date >= CALENDAR_START`

The `signal_date` calendar (end-of-month rebalance dates) is derived from this table.
The `mcap` column here is the authoritative cap-weighting weight for Stages 5–6.

🧪 **VALIDATE:**
- ~330 monthly signal dates (1999-01-29 to present)
- ESTU size: mean ~2,400–2,600 per date; no date with fewer than ~1,700
- `mcap` non-null and positive for all ESTU rows

## §7. STAGE 4 — SIZE estimator

✅ **PDF SPEC (Barra USE4 Empirical Notes, Appendix A, p. 52):**

> *"LNCAP — Log of market capitalization of the firm."*
>
> `SIZE = 1.0 · LNCAP`

The descriptor is a spot computation: for each `(permaticker, signal_date)` pair,
take the most recent market capitalization on or before the signal date and compute
its natural log.

❓ **NOT IN PDF — what counts as "market capitalization."** Sharadar `marketcap`
is in millions of dollars, not dollars.

💡 **DEFAULT:**
```
lncap_{i,t} = ln(marketcap_{i,t*} × 1,000,000)
```
where `marketcap_{i,t*}` is the most recent SEP `marketcap` on or before `signal_date t`
(backward asof join). The `× 1,000,000` converts from $M to $.

❓ **NOT IN PDF — PIT mechanics.** How to join a daily price series to a monthly signal calendar.

💡 **DEFAULT:** Backward asof join on `(permaticker, signal_date ≤ date)`. This picks up
the most recent trading day's `marketcap` that is available as of the signal date. If no
SEP row exists within a reasonable lookback (e.g., the stock hasn't traded recently), the
row is null and the stock receives no `lncap` for that date.

---
*The section above (PDF SPEC quote through final 💡 DEFAULT) is the `STAGE4_DESCRIPTION`
that `/build-factor` will inject verbatim into the Stage 4 stub docstring. Content below
this line is supplementary guidance for the implementer and is not extracted.*
---

**Implementation notes:**
- The asof join should be restricted to a reasonable staleness window — if the most
  recent SEP row is more than ~5 business days before `signal_date`, the price is stale
  and the stock may be suspended. A staleness guard is NOT required by the spec but may
  be considered as a NOT-IN-PDF judgment call.
- After computing `lncap`, join ESTU membership (`in_estu`, `mcap`) from `estu_monthly`.
  Set `in_estu = False` for stocks not in ESTU. Set `n_obs = 1` for all valid rows.
- The `mcap` column in the output comes from `estu_monthly`, not from the SEP asof join.
  For ESTU members they should be very close. For non-ESTU members, `estu_monthly.mcap`
  may differ from the raw SEP value; this is expected and consistent with other factors.

🧪 **VALIDATE:**
- Spot check on a recent signal date: expect ~4,000–6,000 tickers with non-null `lncap`
- `lncap` range: ~[13.8, 29.0] — i.e., ln($1M) ≈ 13.8 to ln($4T) ≈ 29.0
- ESTU median `lncap` ≈ 23.0 (ln of ~$10B)
- Zero duplicate `(permaticker, signal_date)` rows

## §8. STAGE 5 — Outlier treatment

✅ **PDF SPEC** (Methodology Notes §2.2, p. 8):

> *"We trim these observations to three standard deviations from the mean."*

The PDF specifies 3σ with a mean, but does not specify whether the mean is
cap-weighted or equal-weighted, nor the multiplier for factors where the descriptor
is directly correlated with market cap.

❓ **NOT IN PDF — trim center (CW vs EW mean):** For all other USE4 factors the
cap-weighted and equal-weighted means are close, so the trim center does not matter.
For SIZE, `lncap` is directly correlated with market cap: the CW mean of lncap
(≈ 25.5) is structurally ~4 units above the EW mean (≈ 21.5). Using the CW mean
as the trim center places the 3σ lower bound inside the small-cap ESTU body,
clipping ~20% of ESTU rows.

💡 **DEFAULT:** Use the **equal-weighted mean** as the trim center for SIZE. This
correctly centers the trim on the typical ESTU stock rather than the largest stocks.

❓ **NOT IN PDF — σ multiplier:** `lncap` is approximately Gaussian by construction
(log of a log-normal variable). Post-trim excess kurtosis ≈ 0.2 — no heavy tails.
A 3σ EW upper bound (≈ 25.75) clips the large-cap Q5 tail, compressing the quintile
spread below the required 4.0 log-unit validation target.

💡 **DEFAULT:** Use **4σ** (not 3σ). With 4σ, hi ≈ 27.2, above all but the very
largest names. ESTU clip rate drops below 0.1% and the Q5 tail is preserved. This
is defensible because lncap is approximately Gaussian — a 4σ bound on a Gaussian
distribution clips < 0.01% in theory; the observed ~0.05% reflects the slight
positive skew of the empirical distribution.

**Trim formula (per signal_date, ESTU members only):**
```
ew_mean_t = mean_{i ∈ ESTU_t}(lncap_i)
sigma_t   = std_{i ∈ ESTU_t}(lncap_i)
lo_t      = ew_mean_t − 4 × sigma_t
hi_t      = ew_mean_t + 4 × sigma_t
lncap_i   = clip(lncap_i, lo_t, hi_t)   ← applied to ALL stocks
```

🧪 **VALIDATE:**
- ESTU clip rate < 0.3% (lncap is approximately Gaussian; 4σ EW bound is very wide)
- Post-trim ESTU skewness |skew| < 1.0 (expected ≈ 0.7 from the natural right-skew of log-mcap)
- Post-trim ESTU excess kurtosis close to 0 (expected ≈ 0.2)
- lo mean ≈ 15.9 (ln of ~$8M), hi mean ≈ 27.2 (ln of ~$560B) — with EW mean ≈ 21.5, σ ≈ 1.4
- Upper bound hi should exceed Q5 mean lncap (~24) by at least 3 log units

## §9. STAGE 6 — Standardize (cap-weighted mean = 0, EW std = 1)

✅ **PDF SPEC (Methodology Notes §2.3, p. 9):**

> *"μ_l is the cap-weighted mean of the descriptor (within the estimation universe),
> and σ_l is the equal-weighted standard deviation."*

**Per signal_date `t`, using only ESTU members:**

```
μ_t       = Σ_{i ∈ ESTU_t} (mcap_i · lncap_i) / Σ mcap_i   # cap-weighted mean
σ_t       = std_{i ∈ ESTU_t}(lncap_i)                        # equal-weighted std
size_score_{i,t} = (lncap_i − μ_t) / σ_t                     # applied to ALL stocks
```

Three critical points:
1. Moments estimated on **ESTU only**; applied to **every stock** in the coverage universe.
2. Mean is **cap-weighted**; std is **equal-weighted**.
3. Cap-weighting the mean ensures the cap-weighted market portfolio has zero SIZE exposure —
   correct by construction.

**Economic note on the cap-weighted mean:** Large-cap stocks dominate `μ_t`. This is
intentional for SIZE — it centers the factor so that the portfolio-weighted market has
zero exposure, not that the median stock has zero exposure. The median stock will have
negative `size_score` (most stocks by count are small-cap).

Guard: if σ_t < 1e-8, set `size_score = null` for that date (prevents division by zero
on degenerate dates; should never fire in practice for SIZE).

🧪 **VALIDATE:**
- CW mean of `size_score` within ESTU ≈ 0 (|mean| < 1e-6 by construction)
- EW std of `size_score` within ESTU ≈ 1.0 (|std − 1| < 0.02 by construction)
- Fraction of ESTU `size_score` in [−3, +3] > 95%
- Median `size_score` across ESTU will be negative (most stocks by count are small)

## §10. STAGE 7 — Save deliverable

Write the final panel to `data/out/size_use4.parquet`.
Compression: zstd. Statistics: True.

Cast `n_obs` to `UInt32` before writing. Assert every column dtype individually.
Read back from disk and assert shape and column order match.

The raw `lncap` column is included in the output. The downstream WLS step in the
risk model uses `exp(lncap_i / 2) ≈ √(mcap_i)` as the regression weight for
stock `i`. Do not drop or transform `lncap` after standardization.

## §11. STAGE 8 — Validation

### Required checks (all must pass before signing off)

**1. Standardization moments on ESTU:**
```
cap-weighted mean of size_score ≈ 0   (|mean| < 1e-6)
equal-weighted std of size_score ≈ 1   (|std − 1| < 0.02)
```

**2. Raw descriptor sanity:**
```
All ESTU lncap values in [16.0, 30.0]   (ln($8.8M) to ln($10.7T))
Median ESTU lncap in [21.0, 25.0]       (ln($1.3B) to ln($72B))
```

**3. Coverage stability:**
```
≥ 4,000 stocks with non-null size_score per date post-2005
```

**4. Factor stability (month-over-month rank correlation):**
```
MoM Spearman ρ > 0.99   (market cap is highly persistent)
```

**5. Economic direction (quintile structure):**
```
Mean lncap increases monotonically from Q1 to Q5.
Q5 mean lncap ≫ Q1 mean lncap (spread ≥ 3.75 log units, i.e., ≥ 42× in market cap).
Largest-cap names (AAPL, MSFT, NVDA, GOOGL) in Q5.
Micro-cap OTC names in Q1.
```

*Note: the threshold is 3.75 rather than the 4.0 default. For a Gaussian with σ ≈ 1.4
(the observed post-trim EW std for lncap), the theoretical Q1→Q5 quintile-mean spread
ceiling is ≈ 2.6σ ≈ 3.64. The positive skew of log-mcap (≈ 0.7) pushes the observed
spread to ~3.89. A threshold of 3.75 sits below the structural ceiling while still
catching genuine regressions (e.g. collapse back to 3σ CW trim gives spread ≈ 3.49).*

**6. Disk vs. memory equivalence:**
```
max abs diff of lncap values < 1e-10
```

---

**Shared validation conventions (all factors, 2026-06-10):**
- **Coverage (check 3)** is evaluated on **completed months only** — the final
  signal date is the in-progress month (freshest data can lag the Sharadar
  refresh) and is excluded from the floor.
- **Fraction of scores in [−3, +3]** is reported for the full universe *and* for
  ESTU; the ESTU figure is the operationally relevant one (non-ESTU micro-caps
  pull the all-universe tail).
- Common check machinery lives in `common/`, your shared utilities.

## §12. Master list of ❓ NOT-IN-PDF decisions

| # | Decision | Default chosen | Alternative | Where to revisit |
|---|---|---|---|---|
| 1 | Market cap source | `marketcap × 1e6` from `SHARADAR_SEP.csv` | `closeunadj × sharesbas` if `sharesbas` is added to prices in future | Stage 1 — data load |
| 2 | Price column (if computing directly) | `closeunadj` (unadjusted, matches how Sharadar computes `marketcap`) | `close` (same in Sharadar; avoid `closeadj` which retroactively adjusts) | Stage 1 — derivation |
| 3 | Non-positive market cap | Mark `lncap` null; exclude from all computation | Floor at $0.1M (but produces physically meaningless LNCAP for micro OTCs) | Stage 1 filter |
| 4 | Calendar start | 1999-01-01 | Earlier if needed | Stage 0 parameter |
| 5 | MIN_OBS | 1 (any valid spot obs suffices) | Higher threshold if staleness becomes an issue | Stage 0 parameter |
| 6 | n_obs semantics | Always 1 for valid rows; kept for schema consistency | Drop `n_obs` entirely (meaningful for rolling-window factors only) | §3 output schema |
| 7 | Trim center (CW vs EW mean) | **Equal-weighted mean** — CW mean (≈ 25.5) is dominated by large-caps, placing the 3σ lower bound inside the small-cap ESTU body and clipping ~20% of rows | Cap-weighted mean (PDF default for all other factors; inappropriate for SIZE because lncap is directly correlated with market cap) | Stage 5 |
| 8 | Trim σ multiplier | **4σ** — lncap is approximately Gaussian (excess kurtosis ≈ −0.1); a 3σ EW upper bound clips the large-cap Q5 tail and compresses the quintile spread below the 4.0 log-unit validation target | 3σ (PDF default; appropriate when the distribution has heavy tails requiring aggressive clipping) | Stage 5 |

## §13. Common pitfalls — read this before coding

**Pitfall 1: Forgetting the ×1,000,000 unit conversion**
Sharadar `marketcap` is in millions of dollars. Computing `lncap = ln(marketcap)` gives
`ln(dollars / 1,000,000)`, which is ~13.8 units too low everywhere. The standardized
`size_score` will be unaffected (the additive constant cancels in the mean subtraction),
but the raw `lncap` values will be wrong and unusable as WLS weights downstream.
Always use `ln(marketcap × 1_000_000)`.

**Pitfall 2: Using `closeadj` instead of `closeunadj`**
`closeadj` is retroactively adjusted for both splits and dividends. After a dividend
payment, `closeadj` drops relative to `closeunadj`, making `closeadj × sharesbas` too
low (it double-subtracts the dividend). The pre-computed Sharadar `marketcap` column is
always based on `closeunadj`; use it directly to avoid this error.

**Pitfall 3: Do not standardize or transform `lncap` before saving**
The raw `lncap` column must be saved as computed (in ln(dollars) units). The downstream
WLS factor return estimation step uses `w_i = exp(lncap_i / 2) = √(mcap_i)`. If `lncap`
is pre-normalized or scaled, the weights will be wrong. Save both `lncap` (raw) and
`size_score` (standardized) in the output parquet.

**Pitfall 4: Using SF1 `sharesbas` instead of SEP `marketcap`**
Sharadar SF1 `sharesbas` in the ARQ dimension reflects the shares count as of the quarterly
filing date, which may lag the actual current share count by up to three months. The SEP
`marketcap` field reflects the contemporaneous share count as of each trading day and is
the correct PIT source for a monthly-signal factor.

## §14. Final summary — what "done" looks like

**The deliverable is one file:** `data/out/size_use4.parquet`

**Acceptance criteria:**

1. ✅ Schema matches §3 (7 columns, exact dtypes)
2. ✅ All 6 required validation checks in §11 pass within tolerance
3. ✅ ESTU has ~2,500–3,000 names per date, stable over time
4. ✅ Cap-weighted mean of `size_score` is machine-zero for every date in ESTU
5. ✅ Equal-weighted std of `size_score` is 1.0 (to 2 decimal places) for every date in ESTU
6. ✅ All ESTU `lncap` values in [16.0, 30.0]; median ≈ 23.0
7. ✅ Coverage of `size_score` across coverage universe ≥ 4,000 stocks per date post-2005
8. ✅ Month-over-month rank stability ρ > 0.99
9. ✅ All ❓ NOT-IN-PDF decisions in §12 are documented in the code comments

**Once SIZE is built and passes validation, the next steps are:**
- SIZE is a primary factor (not downstream-dependent). **NLS** (`nls_spec.ipynb`) consumes
  this deliverable directly — it reads `lncap` and `size_score` from `size_use4.parquet` —
  so build SIZE before NLS. Other factors that interact with SIZE (e.g., cap-weight
  standardization) consume the ESTU `mcap` column rather than `size_score`.
- The raw `lncap` column is the input to WLS factor return estimation. Confirm that the
  downstream risk model pipeline reads `lncap` (not `size_score`) for constructing
  regression weights.